# Komp. A — AHD sanity check

Reads the AHD NetCDF produced by:

```bash
python -m alptherm_icon.regions build inntal_steinberge
```

and plots the three things that have to look right before we hand the
AHD to Komp. C (1-D convection):

1. **S_G(z)** — heated surface per 100 m bin. For an Alpine region
   this should be a broad hump roughly between valley floor and ridge
   tops, with a long thin tail to the highest peaks.
2. **V_a(z)** — residual atmospheric volume per layer. Must be
   monotonically non-decreasing with z (terrain only opens up as you
   go up) and reach the full layer volume above ridge height.
3. **Cumulative S_G** — hypsographic curve. Should integrate to the
   region area, and its shape tells us whether the region is
   valley-dominated, plateau-like, or peak-dominated.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

REGION = "inntal_steinberge"
PROJECT_ROOT = Path.cwd().resolve().parents[0]
AHD_PATH = PROJECT_ROOT / "data" / "regions" / f"{REGION}_ahd.nc"

if not AHD_PATH.exists():
    raise FileNotFoundError(
        f"AHD not found at {AHD_PATH} — run `python -m alptherm_icon.regions build {REGION}` first."
    )

ds = xr.open_dataset(AHD_PATH)
print(ds)

In [ ]:
z = ds["z"].values
s_g = ds["s_g"].values  # m^2 per bin
v_a = ds["v_a"].values  # m^3 per bin
region_area_km2 = ds.attrs["region_area_m2"] / 1e6

print(f"region:       {ds.attrs['region_name']}")
print(f"area:         {region_area_km2:.1f} km^2")
print(f"bins:         {z.size} x {ds.attrs['bin_height_m']:.0f} m")
print(f"z range:      [{z.min():.0f}, {z.max():.0f}] m")
print(f"S_G sum:      {s_g.sum() / 1e6:.1f} km^2 (should equal region area)")
print(f"S_G peak at:  {z[s_g.argmax()]:.0f} m")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.barh(z, s_g / 1e6, height=ds.attrs["bin_height_m"] * 0.9, color="#6a8caf")
ax.set_xlabel("heated surface S_G  [km^2 per 100 m bin]")
ax.set_ylabel("elevation z  [m]")
ax.set_title(f"AHD — {ds.attrs['region_name']}")
ax.grid(True, alpha=0.3)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(v_a / 1e9, z, color="#b25a3a", lw=1.5)
ax.set_xlabel("residual atmospheric volume V_a  [km^3 per layer]")
ax.set_ylabel("elevation z  [m]")
ax.set_title("V_a(z) — should be monotone non-decreasing")
ax.grid(True, alpha=0.3)

diffs = np.diff(v_a)
if np.any(diffs < -1e-6):
    print(f"WARN: V_a non-monotone at {np.where(diffs < -1e-6)[0]}")
else:
    print("OK: V_a is monotone non-decreasing.")
fig.tight_layout()

In [ ]:
cum_above = (s_g[::-1].cumsum())[::-1] / 1e6
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(cum_above, z, color="#3a7d4e", lw=1.5)
ax.set_xlabel("cumulative heated surface above z  [km^2]")
ax.set_ylabel("elevation z  [m]")
ax.set_title("hypsographic curve")
ax.grid(True, alpha=0.3)
fig.tight_layout()

## What to look for

For the Inntal / Steinberge pilot bbox (placeholder, cuts across the
Inn valley):

- **S_G(z)** peak: expect somewhere in the 1500–2000 m range. A peak
  below 800 m means the bbox is sitting too low (mostly valley floor);
  a peak above 2400 m means it's catching too much of the Karwendel /
  Steinberge crest and not enough valley.
- **V_a(z)** at z = z_min should be near zero and should reach the
  full layer volume (region_area_m2 × 100 m) above the highest peak.
- **Hypsographic curve**: a near-linear shape means broad slope
  distribution; a sharp knee means the region is dominated by either
  the valley or the plateau.

If any of these look off, the placeholder bbox in
`configs/regions/inntal_steinberge.geojson` is the first thing to
refine — the AHD code itself is well-covered by the tests in
`tests/test_ahd.py`.